# 🧠 Week 1 — Pydantic Foundations
> **Goal:** Understand data validation with Pydantic v2 so that AI responses never break your code.

---
### What you will learn
1. Why Pydantic exists and what problem it solves
2. BaseModel — defining typed data schemas
3. Field validators and defaults
4. Nested models (think: LLM returning complex JSON)
5. Parsing raw AI output into safe Python objects
6. Mini-challenge: parse a fake LLM response

# Pydantic

**Pydantic** is a Python library used for **data validation and data parsing**.

In simple terms:  
It makes sure the data your program receives is **correct, structured, and of the right type**.

---

## 🔧 What it does

With Pydantic, you define a **data model**, and it automatically:

- checks data types (like `int`, `str`, `list`)
- converts compatible types (like `"25"` → `25`)
- throws clear errors if data is wrong
- helps handle JSON data safely (very useful for AI responses)

---

## 🧠 Why it’s useful (especially for AI)

When you work with AI models, they often return messy or inconsistent data.  
Pydantic helps you convert that output into **clean Python objects** that your code can trust.

---

## 🚀 In AI context (Pydantic AI)

Libraries like **Pydantic** are heavily used in AI applications to:

- structure LLM outputs  
- prevent broken JSON  
- enforce schemas for agents  

---

## 1 — Install & verify

In [13]:
# Run this once to install dependencies
%pip install pydantic python-dotenv rich --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import pydantic
print(f"Pydantic version: {pydantic.__version__}")

Pydantic version: 2.13.1


## 2 — Why Pydantic? The problem without it

In [16]:
# ---- WITHOUT Pydantic ---- (the painful way)
# Imagine an LLM returned this dict
raw_response = {
    "name": "Alice",
    "age": "twenty-five",   # should be int!  LLM hallucinated a string
    "skills": "Python, ML",  # should be a list!
}

# You would have to manually validate every field...
try:
    age = int(raw_response["age"])   # crashes
except ValueError as e:
    print(f"Crash without Pydantic: {e}")
    print("We have to write custom validation for EVERY field. No thanks.")


Crash without Pydantic: invalid literal for int() with base 10: 'twenty-five'
We have to write custom validation for EVERY field. No thanks.


## 3 — Your first BaseModel

In [17]:
# first baseModel
from pydantic import BaseModel, Field
from typing import List, Optional

class UserProfile(BaseModel):
    name: str
    age: int
    skills: List[str]
    bio: Optional[str] = None  # optional field with default None

# Pydantic auto-coerces types where possible
user = UserProfile(
    name="Alice",
    age="25",           # string "25" → automatically cast to int 25
    skills=["Python", "ML"],
)

print(user)
print(f"Age type: {type(user.age)}")
print(f"Model dict: {user.model_dump()}") 

# user = UserProfile(name="Alice", age=25, skills=["Python"])
# Now:user.model_dump()
# Output:{'name': 'Alice', 'age': 25, 'skills': ['Python'], 'bio': None}



name='Alice' age=25 skills=['Python', 'ML'] bio=None
Age type: <class 'int'>
Model dict: {'name': 'Alice', 'age': 25, 'skills': ['Python', 'ML'], 'bio': None}


## 4 — Validation errors (catching bad AI output)

In [18]:
from pydantic import ValidationError

# Simulate broken LLM output
bad_llm_response = {
    "name": "Bob",
    "age": "twenty-five",   # cannot be cast to int
    "skills": ["Go"],
}

try:
    user = UserProfile(**bad_llm_response)
except ValidationError as e:
    print("Validation failed — here is exactly what went wrong:")
    print(e)
    # You can also iterate errors programmatically
    for error in e.errors():
        print(f"  Field: {error['loc']}  |  Problem: {error['msg']}")

Validation failed — here is exactly what went wrong:
1 validation error for UserProfile
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty-five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
  Field: ('age',)  |  Problem: Input should be a valid integer, unable to parse string as an integer


## 5 — Field constraints & defaults

In [19]:
## 5 — Field constraints & defaults

from pydantic import BaseModel, Field, field_validator
from typing import List

class JobPosting(BaseModel):
    title: str = Field(..., min_length=3, max_length=100, description="Job title")
    salary_usd: int = Field(..., ge=0, le=10_000_000, description="Annual salary in USD")
    required_skills: List[str] = Field(default_factory=list)
    remote: bool = Field(default=False)

    # Custom validator — runs after type coercion
    @field_validator("required_skills")
    @classmethod
    def skills_must_not_be_empty_strings(cls, v: List[str]) -> List[str]:
        cleaned = [s.strip() for s in v if s.strip()]
        if not cleaned:
            raise ValueError("skills list cannot be all empty strings")
        return cleaned

job = JobPosting(
    title="Senior ML Engineer",
    salary_usd=180_000,
    required_skills=[" Python ", "PyTorch", "  "],  # whitespace gets cleaned
    remote=True,
)
print(job)
print(f"Cleaned skills: {job.required_skills}")

title='Senior ML Engineer' salary_usd=180000 required_skills=['Python', 'PyTorch'] remote=True
Cleaned skills: ['Python', 'PyTorch']


## 6 — Nested models (real-world AI output shape)

In [20]:
# 6 nested models 
from pydantic import BaseModel
from typing import List, Optional
from enum import Enum

class SentimentLabel(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL  = "neutral"

class Sentence(BaseModel):
    text: str
    sentiment: SentimentLabel
    confidence: float  # 0.0 – 1.0

class DocumentAnalysis(BaseModel):
    title: str
    overall_sentiment: SentimentLabel
    sentences: List[Sentence]
    summary: Optional[str] = None

# Simulate what an LLM would return as JSON
fake_llm_json = {
    "title": "Product Review",
    "overall_sentiment": "positive",
    "sentences": [
        {"text": "Great product!", "sentiment": "positive", "confidence": 0.97},
        {"text": "Delivery was slow.", "sentiment": "negative", "confidence": 0.84},
    ],
    "summary": "Mixed review, mostly positive.",
}

analysis = DocumentAnalysis(**fake_llm_json)
print(analysis.model_dump_json(indent=2))


{
  "title": "Product Review",
  "overall_sentiment": "positive",
  "sentences": [
    {
      "text": "Great product!",
      "sentiment": "positive",
      "confidence": 0.97
    },
    {
      "text": "Delivery was slow.",
      "sentiment": "negative",
      "confidence": 0.84
    }
  ],
  "summary": "Mixed review, mostly positive."
}


## 7 — Parsing from JSON string (typical AI output)

In [21]:
import json

# LLMs often return a raw JSON string — here is how you safely parse it
raw_json_string = '''
{
  "title": "Tech Blog Post",
  "overall_sentiment": "neutral",
  "sentences": [
    {"text": "Python is popular.", "sentiment": "neutral", "confidence": 0.78}
  ]
}
'''

# Method 1 — via dict
data = json.loads(raw_json_string)
analysis = DocumentAnalysis(**data)

# Method 2 — direct model_validate_json (Pydantic v2)
analysis2 = DocumentAnalysis.model_validate_json(raw_json_string)

print(analysis2.title)
print(f"Sentences parsed: {len(analysis2.sentences)}") #Pydantic successfully found 1 sentence in the LLM output

Tech Blog Post
Sentences parsed: 1


## 8 — JSON Schema generation (used by PydanticAI to instruct the LLM)

In [22]:
import json

# PydanticAI feeds this schema to the LLM so it knows what JSON to produce
schema = DocumentAnalysis.model_json_schema()
print(json.dumps(schema, indent=2))

{
  "$defs": {
    "Sentence": {
      "properties": {
        "text": {
          "title": "Text",
          "type": "string"
        },
        "sentiment": {
          "$ref": "#/$defs/SentimentLabel"
        },
        "confidence": {
          "title": "Confidence",
          "type": "number"
        }
      },
      "required": [
        "text",
        "sentiment",
        "confidence"
      ],
      "title": "Sentence",
      "type": "object"
    },
    "SentimentLabel": {
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "title": "SentimentLabel",
      "type": "string"
    }
  },
  "properties": {
    "title": {
      "title": "Title",
      "type": "string"
    },
    "overall_sentiment": {
      "$ref": "#/$defs/SentimentLabel"
    },
    "sentences": {
      "items": {
        "$ref": "#/$defs/Sentence"
      },
      "title": "Sentences",
      "type": "array"
    },
    "summary": {
      "anyOf": [
        {
          "type": "stri

## 🏋️ Mini Challenge — Week 1
Define a `ResumeAnalysis` model that an LLM could return with:
- `candidate_name: str`
- `years_experience: int` (must be >= 0)
- `top_skills: List[str]` (at least 1 skill)
- `hire_recommendation: bool`
- `notes: Optional[str]`

Then parse the following fake LLM response and print the result.

In [ ]:
# YOUR SOLUTION HERE
fake_resume_response = '''
{
  "candidate_name": "Carol",
  "years_experience": "3",
  "top_skills": ["Python", "FastAPI", "Docker"],
  "hire_recommendation": true,
  "notes": "Strong backend candidate."
}
'''

# TODO: define ResumeAnalysis and parse fake_resume_response


## Week 1 — Summary
| Concept | Key Takeaway |
|---|---|
| `BaseModel` | Defines the shape of data |
| Type coercion | Pydantic auto-casts compatible types |
| `ValidationError` | Catch broken AI output safely |
| `Field(...)` | Add constraints (min, max, regex) |
| `@field_validator` | Custom validation logic |
| Nested models | Model complex LLM JSON responses |
| `model_json_schema()` | Feed schema to LLM as instructions |

**Next: Week 2 — PydanticAI Core → actual AI agents with typed outputs**